# Assignment 2 - LLM Document Analysis of SEC 8-K Filings
**Course:** CAP5619 | **Author:** Eduardo Salcedo

**Pipeline:** S&P 500 tickers -> CIKs -> EDGAR 8-K Atom feed -> filing documents (8-K body + EX-99 press release) -> relevant passages -> local LLM (Ollama) with NER-style prompt -> validated rows -> CSV.

**Before running:**
1. Install Ollama and pull a model: `ollama pull llama3.1` (start the app / `ollama serve`).
2. `pip install requests beautifulsoup4 pandas lxml`
3. Edit `USER_AGENT` below (SEC requires a name and contact email on every request).

In [1]:
import csv, json, re, time, os
import xml.etree.ElementTree as ET
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------------- Configuration ----------------
USER_AGENT   = "ed673399@ucf.edu"   # <-- EDIT: SEC requires a real contact
OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL        = "llama3.1"        # any local model you pulled (e.g. "deepseek-r1:8b", "llama3.2")
TARGET_ROWS  = 110               # stop once we have this many product rows (>=100 for full credit)
MAX_DOCS     = 600               # safety cap on number of 8-K filings sent to the LLM
FILINGS_PER_COMPANY = 15         # 8-Ks pulled per company from the feed
MAX_CHARS_TO_LLM    = 6000       # text budget per filing sent to the LLM
OUTPUT_CSV   = "new_products_8k.csv"
OUTPUT_PIPE  = "new_products_8k.txt"   # pipe-separated table in the assignment's format
RAW_LOG      = "llm_raw_responses.jsonl"  # audit trail of every LLM answer
SEC_DELAY    = 0.15              # seconds between SEC requests (limit is 10/sec)

HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}
COLUMNS = ["company_name", "stock_name", "filing_time", "new_product", "product_description"]

## 1. Company universe (S&P 500) and CIK lookup

In [2]:
def sec_get(url, **kw):
    """GET from SEC with required headers, polite delay, and basic retry."""
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30, **kw)
            time.sleep(SEC_DELAY)
            if r.status_code == 200:
                return r
            if r.status_code in (429, 503):
                time.sleep(2 * (attempt + 1))
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    return None


def load_sp500_tickers():
    """S&P 500 tickers from Wikipedia; falls back to the first 150 SEC tickers if that fails."""
    try:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                            headers={"User-Agent": USER_AGENT}, timeout=30).text
        df = pd.read_html(StringIO(html))[0]
        return [t.replace(".", "-") for t in df["Symbol"].tolist()]
    except Exception as e:
        print("Wikipedia list failed, using SEC ticker order instead:", e)
        return None


def load_cik_map():
    """ticker -> zero-padded 10-digit CIK, from SEC's company_tickers.json."""
    data = sec_get("https://www.sec.gov/files/company_tickers.json").json()
    return {v["ticker"].upper(): str(v["cik_str"]).zfill(10) for v in data.values()}, \
           [v["ticker"].upper() for v in data.values()]

In [3]:
cik_map, sec_order = load_cik_map()
tickers = load_sp500_tickers() or sec_order[:150]
tickers = [t for t in tickers if t.upper() in cik_map]
print(f"{len(tickers)} tickers with a CIK. Example:", tickers[:5])

503 tickers with a CIK. Example: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']


## 2. Find 8-K filings (EDGAR Atom feed) and fetch their documents

In [4]:
def _local(tag):
    return tag.rsplit("}", 1)[-1]


def parse_filing_feed(xml_text):
    """Parse EDGAR Atom feed -> list of dicts (accession, filing_date, index_url, items, company)."""
    out = []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        return out
    for entry in root.iter():
        if _local(entry.tag) != "entry":
            continue
        rec = {}
        for el in entry.iter():
            name = _local(el.tag)
            if name in ("accession-number", "filing-date", "filing-href", "items-desc") and el.text:
                rec[name] = el.text.strip()
            elif name == "title" and el.text:
                rec["title"] = el.text.strip()
        if "filing-href" in rec:
            m = re.match(r"8-K\w*\s*-\s*(.+?)\s*\(", rec.get("title", ""))
            rec["company"] = m.group(1) if m else ""
            out.append(rec)
    return out


def get_8k_filings(ticker, count=FILINGS_PER_COMPANY):
    cik = cik_map[ticker.upper()]
    url = (f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}"
           f"&type=8-K&count={count}&output=atom")
    r = sec_get(url)
    return parse_filing_feed(r.text) if r else []


def parse_index_documents(index_html, base="https://www.sec.gov"):
    """From a filing index page, return URLs of the 8-K body and EX-99.* exhibits (press releases)."""
    soup = BeautifulSoup(index_html, "html.parser")
    urls = []
    for row in soup.select("table.tableFile tr"):
        cells = row.find_all("td")
        if len(cells) < 4:
            continue
        doc_type = cells[3].get_text(strip=True).upper()
        a = cells[2].find("a")
        if not a or not (doc_type.startswith("8-K") or doc_type.startswith("EX-99")):
            continue
        href = a["href"]
        if "doc=" in href:                      # inline-XBRL viewer link -> raw document
            href = href.split("doc=", 1)[1]
        if href.lower().endswith((".htm", ".html", ".txt")):
            urls.append(base + href if href.startswith("/") else href)
    return urls


def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style"]):
        t.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ")).strip()


def fetch_filing_text(index_url):
    r = sec_get(index_url)
    if not r:
        return ""
    texts = []
    for u in parse_index_documents(r.text)[:3]:
        d = sec_get(u)
        if d:
            texts.append(html_to_text(d.text))
    return " ".join(texts)

## 3. Pre-filter passages
8-Ks cover many events (earnings, officer changes, debt offerings). To save LLM time we keep only passages around product-launch language, then let the LLM decide what is actually a *new product*.

In [5]:
PRODUCT_PATTERN = re.compile(
    r"\b(launch\w*|introduc\w*|unveil\w*|announc\w*|releas\w*|new product|now available|"
    r"rolls? out|debut\w*|general availability)\b", re.I)


def relevant_passages(text, window=700, budget=MAX_CHARS_TO_LLM):
    """Return merged text windows around product-release keywords, capped at `budget` chars."""
    spans = []
    for m in PRODUCT_PATTERN.finditer(text):
        s, e = max(0, m.start() - window // 2), min(len(text), m.end() + window)
        if spans and s <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], e)
        else:
            spans.append([s, e])
    snippet = " ... ".join(text[s:e] for s, e in spans)
    return (snippet or text)[:budget]

## 4. LLM extraction (Ollama, NER-style prompt)

In [6]:
PROMPT = """You are an information-extraction system doing Named Entity Recognition on SEC Form 8-K text.

Task: find NEW PRODUCTS that the company announces or releases in the text below.
- Entity types: ORGANIZATION = the filing company; PRODUCT = a newly announced or launched product, service, drug, device or platform.
- Do NOT list earnings, financings, acquisitions, executives, existing products merely mentioned, or events.
- If there is no new product, return an empty list.
- product_description: factual, under 150 characters.

Return ONLY valid JSON in exactly this shape:
{"company_name": "<ORGANIZATION name as written in the text>",
 "products": [{"new_product": "<PRODUCT name>", "product_description": "<short description>"}]}

TEXT:
<<TEXT>>"""


def ask_llm(text, model=MODEL):
    """Send text to local Ollama (JSON mode, temperature 0). Returns parsed dict or None."""
    payload = {"model": model, "prompt": PROMPT.replace("<<TEXT>>", text),
               "stream": False, "format": "json",
               "options": {"temperature": 0, "num_ctx": 8192}}
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=300)
        raw = r.json().get("response", "")
    except Exception as e:
        print("  LLM error:", e)
        return None
    with open(RAW_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({"raw": raw}) + "\n")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.S)       # DeepSeek-style models may wrap JSON in prose
        try:
            return json.loads(m.group(0)) if m else None
        except Exception:
            return None


def clip_description(desc, limit=180):
    """Hard-enforce the <180 character rule, cutting at a word boundary."""
    desc = re.sub(r"\s+", " ", str(desc or "")).strip()
    if len(desc) < limit:
        return desc
    return desc[: limit - 4].rsplit(" ", 1)[0].rstrip(",;:") + "..."


def to_rows(llm_json, ticker, filing, fallback_company):
    """Validate LLM output and convert to CSV rows."""
    rows = []
    if not isinstance(llm_json, dict):
        return rows
    company = (llm_json.get("company_name") or fallback_company or "").strip()
    for p in llm_json.get("products") or []:
        name = str((p or {}).get("new_product", "")).strip()
        if not name or name.lower() in ("none", "n/a", "null", "unknown"):
            continue
        rows.append({
            "company_name": company or fallback_company,
            "stock_name": ticker,
            "filing_time": filing.get("filing-date", ""),
            "new_product": name,
            "product_description": clip_description(p.get("product_description")),
        })
    return rows

## 5. Run the pipeline (saves after every row, so you can stop and resume)

In [7]:
def save_outputs(rows):
    df = pd.DataFrame(rows, columns=COLUMNS).drop_duplicates(["stock_name", "filing_time", "new_product"])
    df.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)
    with open(OUTPUT_PIPE, "w", encoding="utf-8") as f:
        f.write("Company Name | Stock Name | Filing Time | New Product | Product Description\n")
        f.write("-" * 90 + "\n")
        for r in df.itertuples(index=False):
            f.write(" | ".join(map(str, r)) + "\n")
    return df


def run_pipeline():
    rows, docs_processed, companies_seen = [], 0, set()
    for ticker in tickers:
        if len(rows) >= TARGET_ROWS or docs_processed >= MAX_DOCS:
            break
        for filing in get_8k_filings(ticker):
            if len(rows) >= TARGET_ROWS or docs_processed >= MAX_DOCS:
                break
            text = fetch_filing_text(filing["filing-href"])
            if len(text) < 200:
                continue
            docs_processed += 1
            companies_seen.add(ticker)
            result = ask_llm(relevant_passages(text))
            new_rows = to_rows(result, ticker, filing, filing.get("company", ""))
            rows.extend(new_rows)
            print(f"[{docs_processed:>3} docs | {len(rows):>3} rows] {ticker} {filing.get('filing-date','')}: "
                  f"{[r['new_product'] for r in new_rows] or '-'}")
            if new_rows:
                save_outputs(rows)
    df = save_outputs(rows)
    return df, docs_processed, len(companies_seen)


df, n_docs, n_companies = run_pipeline()
print(f"\nDone. Filings analyzed: {n_docs} | Companies: {n_companies} | Product rows: {len(df)}")

[  1 docs |   2 rows] MMM 2026-11-04: ['state-of-the-art water treatment technologies', 'PFAS remediation technologies']
[  2 docs |   4 rows] MMM 2026-10-20: ['Cooling Film', 'Expanded Beam Optics (EBO) solutions']
[  3 docs |   5 rows] MMM 2026-10-19: ['optical connectivity and other Data Center products']
[  4 docs |   5 rows] MMM 2026-09-10: -
[  5 docs |   5 rows] MMM 2026-08-19: -
[  6 docs |   8 rows] MMM 2026-07-21: ['Expanded Beam Optics (EBO) technology', 'Ask 3M', 'PELTOR ComTac VI Tactical Headsets']
[  7 docs |   8 rows] MMM 2026-06-08: -
[  8 docs |   8 rows] MMM 2026-05-13: -
[  9 docs |   9 rows] MMM 2026-05-06: ['Fire Safety Platform']
[ 10 docs |  10 rows] MMM 2026-04-21: ['new product and service offerings']
[ 11 docs |  13 rows] AOS 2026-10-29: ['Leonard Valve', 'boiler', 'China business']
[ 12 docs |  14 rows] AOS 2026-10-28: ['China business']
[ 13 docs |  15 rows] AOS 2026-07-30: ['Leonard Valve']
[ 14 docs |  16 rows] AOS 2026-06-22: ["A. O. Smith Corporation's 

## 6. Inspect results (use these numbers in the report)

In [8]:
df = pd.read_csv(OUTPUT_CSV)
print("Rows:", len(df), "| Companies:", df.stock_name.nunique(),
      "| Max description length:", df.product_description.str.len().max())
df.head(15)

Rows: 116 | Companies: 7 | Max description length: 137


,company_name,stock_name,filing_time,new_product,product_description
0,3M Company,MMM,2026-11-04,state-of-the-art water treatment technologies,water treatment technologies at sites where PF...
1,3M Company,MMM,2026-11-04,PFAS remediation technologies,technologies to remediate PFAS at sites where ...
2,3M Company,MMM,2026-10-20,Cooling Film,Thermal management technology for fleet and ra...
3,3M Company,MMM,2026-10-20,Expanded Beam Optics (EBO) solutions,Optical interconnect expertise and materials s...
4,3M Company,MMM,2026-10-19,optical connectivity and other Data Center pro...,products for Amazon's data centers
5,3M Company,MMM,2026-07-21,Expanded Beam Optics (EBO) technology,AI data center infrastructure and enterprise t...
6,3M Company,MMM,2026-07-21,Ask 3M,AI digital assistant powered by AWS for techni...
7,3M Company,MMM,2026-07-21,PELTOR ComTac VI Tactical Headsets,Communication headsets for extreme environments
8,3M Company,MMM,2026-05-06,Fire Safety Platform,a credit agreement for a term loan facility an...
9,3M Company,MMM,2026-04-21,new product and service offerings,new product and service offerings
